In [ ]:
import re
import numpy as np
import pandas as pd
# load
df = pd.read_csv(r"C:\Users\Ameen1\Desktop\Test-code\anime_data.csv").dropna(subset=["mean"]).copy()

TITLE_COL  = "title"
GENRE_COL  = "genres"     
STUDIO_COL = "studios"  

STOPWORDS = {"the","a","an","and","of","to","in","on","for","with","no"}

REMOVE_WORDS = r"(season|part|cour|movie|film|ova|ona|special|recap|tv)"

# helpers
def clean_title_tokens(title: str):
    t = str(title).lower()
    t = re.sub(r"\(.*?\)|\[.*?\]|\{.*?\}", " ", t)            # remove bracketed text
    t = re.sub(r"[^a-z0-9]+", " ", t)                         # keep letters/numbers
    t = re.sub(rf"\b{REMOVE_WORDS}\b", " ", t)                # remove format words
    t = re.sub(r"\b\d+(st|nd|rd|th)?\b", " ", t)              # remove numbers like 2, 2nd
    t = re.sub(r"\b(ii|iii|iv|v|vi|vii|viii|ix|x)\b", " ", t) # remove roman numerals
    t = re.sub(r"\s+", " ", t).strip()

    toks = [w for w in t.split() if w and w not in STOPWORDS]
    return toks

def root_key(title: str):
    toks = clean_title_tokens(title)
    if len(toks) >= 2:
        return f"{toks[0]} {toks[1]}"     # first 2 meaningful words
    return toks[0] if toks else ""

def to_set_list(x):
    if pd.isna(x): 
        return set()
    return {p.strip() for p in str(x).split(",") if p.strip()}

MAX_GENRE_DIFF = 3
def genre_similar(a: set, b: set) -> bool:
    return len(a ^ b) <= MAX_GENRE_DIFF   # symmetric difference <= 3

def studio_overlap(a: set, b: set) -> bool:
    # if missing, don’t block merging
    if not a or not b:
        return True
    return len(a & b) > 0

def merge_genres(genre_sets):
    out = set()
    for s in genre_sets:
        out |= s
    return ", ".join(sorted(out))

# build keys + sets
df["root_key"]   = df[TITLE_COL].apply(root_key)
df = df[df["root_key"] != ""].copy()

df["_gset"] = df[GENRE_COL].apply(to_set_list)
df["_sset"] = df[STUDIO_COL].apply(to_set_list)

# cluster inside each root_key 
final_key = {}
for rk, g in df.groupby("root_key"):
    reps = []  # representative (genre_set, studio_set) for each cluster
    for idx, row in g.iterrows():
        gs, ss = row["_gset"], row["_sset"]

        placed = None
        for i, (rgs, rss) in enumerate(reps):
            if genre_similar(gs, rgs) and studio_overlap(ss, rss):
                placed = i
                break

        if placed is None:
            reps.append((gs, ss))
            placed = len(reps) - 1

        final_key[idx] = f"{rk}__{placed}"

df["franchise_key"] = df.index.map(final_key.get)

# aggregate to 1 row per franchise
fr = (df.groupby("franchise_key", as_index=False)
        .agg(
            franchise_title=(TITLE_COL, "first"),
            mean=("mean", "mean"),                           # average of means
            num_scoring_users=("num_scoring_users", "max"),   # max votes
            genres=("_gset", merge_genres),                  # union genres
        ))

# rank using vote-weighted score 
C = fr["mean"].mean()
m = fr["num_scoring_users"].quantile(0.80)

v = fr["num_scoring_users"].fillna(0)
R = fr["mean"].fillna(C)

fr["score"] = (v / (v + m)) * R + (m / (v + m)) * C
fr = fr.sort_values("score", ascending=False).reset_index(drop=True)
fr["rank"] = np.arange(1, len(fr) + 1)

#  save 
# out_path = r"C:\Users\Ameen1\Desktop\Test-code\anime_franchises_prepped_5.csv"
# fr.to_csv(out_path, index=False)
# print("Saved:", len(fr), "franchises ->", out_path)

# check: Dragon Ball should now be ONE root key ("dragon ball")
mask = df[TITLE_COL].str.contains("dragon ball", case=False, na=False)
print(df.loc[mask, [TITLE_COL, "root_key", "franchise_key"]].drop_duplicates().head(60))